In [2]:
!uv add dotenv

Resolved 147 packages in 1.07s                                       
Prepared 1 package in 35ms                                                   dotenv               ------------------------------ 1.85 KiB/1.85 KiB           
Installed 1 package in 4ms                                  
 + dotenv==0.9.9


In [3]:
from dotenv import load_dotenv
import os
load_dotenv()

GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")

In [6]:
!uv add langchain[google-genai]

Resolved 154 packages in 727ms                                       
Prepared 7 packages in 197ms                                                 cryptography           ------------------------------ 2.79 MiB/4.53 MiB         cryptography           ------------------------------ 1.01 MiB/4.53 MiB         cryptography           ------------------------------ 192.00 KiB/4.53 MiB       cryptography           ------------------------------ 160.00 KiB/4.53 MiB       cryptography           ------------------------------     0 B/4.53 MiB          
Installed 7 packages in 7ms                                 
 + cryptography==50.0.1
 + filetype==1.2.0
 + google-auth==2.58.1
 + google-genai==2.25.0
 + langchain-google-genai==4.4.0
 + pyasn1==0.6.4
 + pyasn1-modules==0.4.2


In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import OllamaEmbeddings

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite", temperature=0.2, api_key=GOOGLE_API_KEY
)
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe:latest")

In [16]:
result = llm.invoke("What is a platypus?")

In [10]:
print(result.text)

A **platypus** (*Ornithorhynchus anatinus*) is a semi-aquatic mammal native to eastern Australia and Tasmania. It is widely considered one of the strangest and most unique animals on Earth because it combines physical traits from birds, reptiles, and mammals. 

When European naturalists first examined a preserved platypus body in 1799, they genuinely believed it was a taxidermy hoax made by sewing parts of different animals together.

Here is what makes the platypus so unusual:

### 1. It is a Monotreme (An Egg-Laying Mammal)
The platypus belongs to an ancient and tiny group of mammals called **monotremes** (the only other living monotremes are four species of echidnas). 
* Unlike almost all other mammals, platypuses **do not give birth to live young—they lay eggs**.
* However, they are still true mammals: they have fur, are warm-blooded, and produce milk to feed their young. 
* Strangely, mothers **do not have nipples**. Instead, milk sweats out of specialized mammary glands on their 

In [22]:
from langchain_community.document_loaders import TextLoader
import json

with open("booklist.json", "r") as f:
    booklist = json.load(f)

documents = []
for book in booklist:
    loader = TextLoader(book["path"], encoding="utf-8")
    doc = loader.load()[0]
    doc.metadata["title"] = book["title"]
    doc.metadata["author"] = book["author"]
    doc.metadata["year"] = book["year"]
    doc.metadata["genre"] = book["genre"]
    doc.metadata["language"] = book["language"]
    documents.extend([doc])

print(f"Loaded {len(documents)} documents from {len(booklist)} books.")

Loaded 2 documents from 2 books.


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=2500, chunk_overlap=200)
chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")

Split into 513 chunks.


In [24]:
from langchain_community.vectorstores import InMemoryVectorStore

vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)

In [25]:
from langchain.tools import tool


@tool
def search_books(query: str) -> str:
    """
    Search context from the book collection in the VectorStore and return the most relevant chunks.
    Example: "Who is Alice? How does she know the Cheshire Cat? What is the story about?"
    """
    results = vectorstore.similarity_search(query, k=3)
    return "\n\n".join(
        [
            f"Title: {doc.metadata['title']}\nAuthor: {doc.metadata['author']}\nYear: {doc.metadata['year']}\nGenre: {doc.metadata['genre']}\nLanguage: {doc.metadata['language']}\nContent: {doc.page_content}"
            for doc in results
        ]
    )

In [26]:
system_prompt = """
## System Prompt: Book Context Retrieval Agent

**Role**

You are a knowledgeable book assistant. When a user asks a question, you retrieve relevant context about the requested book using your available tools, then provide a helpful, 
accurate, and well-grounded answer.

**Available Tool**

You have access to a retrieval tool that returns context about books. Use it whenever a question concerns a book, its plot, characters, themes, quotes, author, publication details, or 
similar topics. Query it with enough detail to retrieve the relevant passage. If the book title is ambiguous, disambiguate before searching.

*Adapt the tool name and parameters below to match your actual interface, e.g.:*
```
search_books(query) → context
```

**Operating Principles**

1. **Ground your answers in retrieved context.** Base every factual claim on what the tool returns. Do not fabricate details, quotes, characters, or events.
2. **Be transparent about limits.** If the retrieved context does not contain the answer, say so honestly rather than guessing.
3. **Stay on topic.** Answer what was asked. Avoid unrelated summaries or plot points unless they directly support the answer.
4. **Cite your sources.** Attribute claims to the specific context or passage when helpful (e.g., "According to the retrieved passage...").
5. **Be concise and clear.** Provide the answer directly; prioritize usefulness over length.
6. **Handle multiple books gracefully.** If a query could refer to more than one book, identify which one the user means, or address both if that's what they asked for.

**Workflow**

1. **Understand the question.** Identify the specific book (if any) and what aspect is being asked about.
2. **Retrieve context.** Call the tool with a focused query. If the first result is insufficient, refine the query and search again—try synonyms, alternate titles, character names, or 
plot keywords.
3. **Evaluate the results.** Check whether the retrieved context actually addresses the question. If it's off-topic or too vague, re-query.
4. **Synthesize an answer.** Compose a response using only the relevant retrieved information, organized for clarity.
5. **Report gaps.** If the context is incomplete for what was asked, tell the user what's missing and, where reasonable, offer the closest available information.

**Handling Retrieval Outcomes**

- **Relevant context found:** Use it to answer directly.
- **Partial match:** Answer the portion the context covers, then note that other parts are unavailable.
- **No relevant context:** Clearly state that the tool did not return usable information, and avoid inventing an answer.
- **Contradictory context:** Present the competing information and, if possible, indicate which appears more consistent with the source.
- **Ambiguous query:** Ask a brief clarifying question (e.g., which book, or which element—plot, character, author) when doing so prevents a wrong answer.

**Answering Guidelines**

- Prefer direct quotes or paraphrases from the retrieved passage for factual claims.
- For subjective questions (interpretations, recommendations), you may draw on reasoning, but still anchor concrete details in context.
- If the question asks for something the context cannot support (e.g., a plot point not present, or an out-of-scope topic), respond accordingly rather than stretching the answer.

**Prohibited Behaviors**

- Do not invent books, characters, quotes, events, or details not present in the retrieved context.
- Do not claim certainty about information the context does not support.
- Do not present guesses as facts.
- Do not ignore the tool when a book question is asked.
"""

In [27]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[search_books], system_prompt=system_prompt)

In [28]:
from langchain.messages import HumanMessage, AIMessage

query = "Who is Alice? How does she know the Cheshire Cat? What is the story about?"


def ask_chat(query: str) -> AIMessage:
    stream = agent.stream_events(
        {"messages": [HumanMessage(content=query)]},
        version="v3",
    )

    last_message = None

    for snapshot in stream.values:
        latest_message = snapshot["messages"][-1]
        if latest_message.content:
            if isinstance(latest_message, HumanMessage):
                print(f"User: {latest_message.content}")
            elif isinstance(latest_message, AIMessage):
                print(f"Agent: {latest_message.pretty_repr()}")
        elif latest_message.tool_calls:
            print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

        last_message = latest_message

    return last_message

In [29]:
last_message = ask_chat(query)

User: Who is Alice? How does she know the Cheshire Cat? What is the story about?
Agent: ================================== Ai Message ==================================

[{'type': 'tool_call', 'id': 'call_1299243', 'name': 'search_books', 'args': {'query': "Who is Alice in Alice's Adventures in Wonderland? How does she know the Cheshire Cat? What is the story about?"}}]
Tool Calls:
  search_books (call_1299243)
 Call ID: call_1299243
  Args:
    query: Who is Alice in Alice's Adventures in Wonderland? How does she know the Cheshire Cat? What is the story about?
Agent: ================================== Ai Message ==================================

[{'type': 'text', 'text': '*Alice\'s Adventures in Wonderland*, written by Lewis Carroll and published in 1865, is a classic work of fantasy literature.\n\n### Who is Alice?\nAlice is a young girl who, while sitting by a riverbank with her sister, becomes bored and curious. Her adventure begins when she spots a White Rabbit wearing a waistco

In [30]:
from IPython.display import display, Markdown

display(Markdown(last_message.text))

*Alice's Adventures in Wonderland*, written by Lewis Carroll and published in 1865, is a classic work of fantasy literature.

### Who is Alice?
Alice is a young girl who, while sitting by a riverbank with her sister, becomes bored and curious. Her adventure begins when she spots a White Rabbit wearing a waistcoat and carrying a pocket watch. Driven by curiosity, she follows the rabbit down a large hole, which leads her into the surreal world of Wonderland.

### How does she know the Cheshire Cat?
Alice does not "know" the Cheshire Cat in the traditional sense; rather, she encounters it during her journey through Wonderland. She first meets the Cat while it is sitting on a tree bough, and she approaches it to ask for directions. Throughout the story, the Cat appears and disappears at will, often engaging Alice in cryptic, philosophical conversations. Alice eventually introduces the Cat to the King of Hearts, referring to it as "a friend of mine."

### What is the story about?
The story follows Alice's journey through the nonsensical and dreamlike world of Wonderland. After falling down the rabbit hole, she encounters a variety of bizarre characters and experiences strange transformations. The narrative is episodic, detailing her interactions with creatures like the Caterpillar, the Mad Hatter at a "Mad Tea-Party," and the Queen of Hearts on her croquet ground. The book is widely known for its imaginative play with logic, language, and the absurdity of the situations Alice faces.

In [31]:
query = "Who is Dracula?"

result = ask_chat(query)

User: Who is Dracula?
Agent: ================================== Ai Message ==================================

[{'type': 'tool_call', 'id': 'call_1179009', 'name': 'search_books', 'args': {'query': 'Who is Dracula?'}}]
Tool Calls:
  search_books (call_1179009)
 Call ID: call_1179009
  Args:
    query: Who is Dracula?
Agent: ================================== Ai Message ==================================

[{'type': 'text', 'text': 'Count Dracula is the titular antagonist of Bram Stoker’s 1897 horror novel, *Dracula*.\n\nAccording to the text, he is a figure of ancient and noble lineage, described by Professor Van Helsing as having likely been the "Voivode Dracula" who fought against the Turks. He is characterized as a man of immense intellect, cunning, and bravery, but also as a "wampyr" (vampire). \n\nKey details about him from the novel include:\n*   **Background:** He belongs to a "great and noble race" from the "land beyond the forest" (Transylvania). Historical records suggest that

In [32]:
display(Markdown(result.text))

Count Dracula is the titular antagonist of Bram Stoker’s 1897 horror novel, *Dracula*.

According to the text, he is a figure of ancient and noble lineage, described by Professor Van Helsing as having likely been the "Voivode Dracula" who fought against the Turks. He is characterized as a man of immense intellect, cunning, and bravery, but also as a "wampyr" (vampire). 

Key details about him from the novel include:
*   **Background:** He belongs to a "great and noble race" from the "land beyond the forest" (Transylvania). Historical records suggest that some members of his family were rumored to have had dealings with the "Evil One" and learned dark secrets in the Scholomance.
*   **Nature:** He is a supernatural being who requires earth from his homeland to rest. He is described as a "man-that-was" who possesses an "iron resolution" and a "mighty brain," which he uses to oppose the protagonists.
*   **Reputation:** Local people in the Carpathians are terrified of him; when Jonathan Harker asks about him, they often refuse to speak and cross themselves in fear.